# Qwen3.5-9B Inference — NL → ASP

## 0 · Login & Imports

In [ ]:
from huggingface_hub import login
login('YOUR HUGGINGFACE_TOKEN')

In [ ]:
import os
import json
from pathlib import Path
from datasets import load_dataset
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from transformers import EarlyStoppingCallback
import trl
from trl import SFTTrainer, SFTConfig
import torch
import peft
from peft import LoraConfig


print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)


## 1 · Qwen3 Output Cleaning Utility

In [ ]:
def strip_think_tokens(text: str) -> str:
    """
    Removes Qwen3 chain-of-thought blocks from generated output.

    Handles:
      1. Fully paired <think>...</think> blocks (including empty ones).
      2. Orphaned </think> with no opening tag — keeps only content AFTER it.
      3. No think tokens — returned unchanged.
    """
    if not text:
        return text

    # Case 1: Remove fully paired <think>...</think> blocks
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)

    # Case 2: Handle orphaned </think> — take only the text AFTER it
    if "</think>" in cleaned:
        cleaned = cleaned.split("</think>", 1)[-1]

    return cleaned.strip()

## 2 · Load Fine-Tuned Adapter

In [ ]:
# ── Update this path to your saved adapter / best checkpoint ──────────────
adapter_path = "Path/To/Your/Saved/Adapter"

model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_path,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded from: {adapter_path}")
device_map = getattr(model, 'hf_device_map', None) or getattr(model.base_model, 'hf_device_map', 'N/A')
print(f"Device map: {device_map}")

## 3 · NaiveDecoder 

In [ ]:
class NaiveDecoder:
    """
    Unconstrained decoder: wraps HuggingFace pipeline for greedy text generation.
    Applies strip_think_tokens to clean any residual chain-of-thought artifacts.
    """

    def __init__(self, model, tokenizer: PreTrainedTokenizer):
        self.tokenizer = tokenizer
        self.model = model
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map=self.device,
        )

    def decode(self, prompt: str, max_new_tokens: int = 512, temperature: float = 0) -> str:
        outputs = self.pipeline(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=temperature,
            top_p=None,
            eos_token_id=self.tokenizer.eos_token_id,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        generated_text = outputs[0]['generated_text']
        prediction = generated_text[len(prompt):].strip()

        # Remove trailing EOS tokens if present
        if self.tokenizer.eos_token and self.tokenizer.eos_token in prediction:
            prediction = prediction.split(self.tokenizer.eos_token)[0].strip()

        # Belt-and-suspenders: strip any residual think tokens
        return strip_think_tokens(prediction)


print("NaiveDecoder defined.")

## 4 · Predictor — NL → ASP

In [ ]:
class Predictor:
    """
    Wraps chat-template prompt construction and decoder call.

    apply_chat_template is called with enable_thinking=False to suppress
    Qwen3's <think> block entirely, avoiding double-output artifacts.
    """

    SYSTEM_PROMPT = (
        "You are an expert in translating Natural Language (NL) into "
        "Answer Set Programming (ASP). Always provide precise, syntactically "
        "and semantically correct ASP translations."
    )

    def __init__(self, decoder: NaiveDecoder, tokenizer: PreTrainedTokenizer):
        self.decoder = decoder
        self.tokenizer = tokenizer

    def predict(self, nl_input: str, max_new_tokens: int = 512) -> str:
        """Build the chat prompt and run the decoder. Returns clean ASP string."""
        prompt = self.tokenizer.apply_chat_template(
            [
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": f"Translate the following natural language to Answer Set Programming: {nl_input}",
                },
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,   # suppresses <think> block entirely
        )
        return self.decoder.decode(prompt, max_new_tokens=max_new_tokens, temperature=0.1)


print("Predictor defined.")

## 5 · NLToASP—Inference

In [ ]:
class NLToASPRunner:
    """
    Batch inference runner: NL → Predicted ASP → CSV.

    Supports checkpoint resumption: if output_filename already exists,
    previously processed samples are skipped and new results are appended.
    """

    def __init__(self, predictor: Predictor):
        self.predictor = predictor

    # ── Public API ────────────────────────────────────────────────────────
    def run_and_save(
        self,
        json_path: str,
        output_filename: str = "nl_to_asp_testset_with_predicates_results_withoutidNLFT.csv",
        verbose: bool = True,
    ) -> list[dict]:
        """
        Run inference over all samples in json_path and save to output_filename.

        Args:
            json_path:       Path to the test JSON file.
            output_filename: Path for the output CSV.
            verbose:         Print progress and summary.

        Returns:
            List of result dicts (same rows as the CSV).
        """
        samples = self._load_samples(json_path)
        total   = len(samples)
        print(f"Loaded {total} samples from: {json_path}")

        # ── Checkpoint: skip already-processed samples ────────────────────
        already_done = self._load_checkpoint(output_filename)
        start_idx    = len(already_done)
        results      = already_done[:]

        if start_idx > 0:
            print(f"Resuming from sample {start_idx} (checkpoint found).")

        # ── Main inference loop ───────────────────────────────────────────
        for item in tqdm(samples[start_idx:], initial=start_idx, total=total,
                         desc="NL → ASP", disable=not verbose):
            nl_input    = item.get("NL_V2", "")
            actual_asp  = item.get("ASP", "")
            category    = item.get("Category", "N/A")
            item_id     = item.get("ID", "N/A")

            predicted_asp = self.predictor.predict(nl_input)

            results.append({
                "ID":                item_id,
                "Category":          category,
                "Natural Language":  nl_input,
                "Predicted ASP":     predicted_asp,
                "Actual ASP":        actual_asp,
            })

        # ── Save ──────────────────────────────────────────────────────────
        df = pd.DataFrame(results)
        os.makedirs(os.path.dirname(output_filename), exist_ok=True) if os.path.dirname(output_filename) else None
        df.to_csv(output_filename, index=False)

        abs_path = os.path.abspath(output_filename)
        if verbose:
            print("\n" + "=" * 45)
            print("INFERENCE COMPLETE")
            print("=" * 45)
            print(f"Total samples : {total}")
            print(f"CSV saved to  : {abs_path}")
            print("=" * 45)

        return results

    # ── Helpers ───────────────────────────────────────────────────────────
    @staticmethod
    def _load_samples(json_path: str) -> list[dict]:
        with open(json_path, "r") as f:
            raw = json.load(f)
        if isinstance(raw, dict) and "data_dict" in raw:
            return raw["data_dict"]
        if isinstance(raw, list):
            return raw
        raise ValueError(f"Unexpected JSON structure in {json_path}")

    @staticmethod
    def _load_checkpoint(output_filename: str) -> list[dict]:
        """If the output CSV already exists, load it as a list of dicts (checkpoint)."""
        if os.path.exists(output_filename):
            return pd.read_csv(output_filename).to_dict(orient="records")
        return []


print("✓ NLToASPRunner defined.")

## 6 · Single-Sample Test

In [ ]:
single_NL = "Give Your Test NL Here."

decoder   = NaiveDecoder(model=model, tokenizer=tokenizer)
predictor = Predictor(decoder=decoder, tokenizer=tokenizer)

predicted_asp = predictor.predict(single_nl)

print("NL Input      :", single_nl)
print("Predicted ASP :", predicted_asp)

## 7 ·  Run Full Evaluation Pipeline

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────
DATASET_FILE = "Path/To/Your_INPUT/Test_Dataset.json"
OUTPUT_FILE  = "Path/To/Your_OUTPUT/outputfile.csv"

# ── Run ───────────────────────────────────────────────────────────────────``
decoder   = NaiveDecoder(model=model, tokenizer=tokenizer)
predictor = Predictor(decoder=decoder, tokenizer=tokenizer)
runner    = NLToASPRunner(predictor=predictor)

results = runner.run_and_save(
    json_path=DATASET_FILE,
    output_filename=OUTPUT_FILE,
    verbose=True,
)